# Tutorial 6: Migrating from Legacy Format

This tutorial covers migrating data and models from the legacy deepSCENIC format to the new unified format.

## Learning Objectives

- Understand the differences between legacy and new formats
- Convert legacy data files to MuData
- Load legacy trained models
- Load GRN matrices for analysis
- Save data and models in the new format

## Setup

In [ ]:
import deepscenic as ds
import pandas as pd

print(f"deepSCENIC version: {ds.__version__}")

## Legacy vs New Format

### Legacy Format (Multiple Files)

The legacy format stores data and models across multiple separate files:

**Data files:**
- `raw_exprMat_train.h5ad` - scRNA-seq expression matrix
- `fragment_matrix_train.h5ad` - scATAC-seq accessibility matrix
- `r2gpenalty_train.npz` - region-to-gene distance penalties (sparse)
- `TFs.txt` - list of transcription factor gene names

**Model files:**
- `best_model.pth` - VAE checkpoint (encoder, decoders, E2 weights)
- `best_model_tf2r.pth` - MotifNet checkpoint
- `best_model_tf2r_encoder.pth` - Enformer checkpoint (~1GB)
- `E1.pkl` - pre-computed TF→region matrix
- `E2.pkl` - pre-computed region→gene matrix

### New Format (Unified)

**Data:** Single `.h5mu` MuData file containing:
- RNA and ATAC modalities
- TF annotations in `mdata.mod["rna"].var["is_tf"]`
- R2G matrix in `mdata.uns["r2g"]`

**Model:** Single `.pt` file containing:
- All model weights (VAE, MotifNet, Enformer)
- Pre-computed E1 matrix
- Feature names (genes, regions, TFs)
- Training configuration

## Step 1: Loading Legacy Data

Use `ds.io.read_legacy()` to convert legacy data files to the new MuData format:

In [ ]:
# Set your legacy data directory
data_dir = "path/to/legacy/data"  # e.g., "/data/MM_lines"

# Load legacy data into MuData format
# mdata = ds.io.read_legacy(
#     rna_path=f"{data_dir}/raw_exprMat_train.h5ad",
#     atac_path=f"{data_dir}/fragment_matrix_train.h5ad",
#     r2g_path=f"{data_dir}/r2gpenalty_train.npz",
#     tf_list_path=f"{data_dir}/TFs.txt",
# )
# 
# print(f"Loaded MuData: {mdata.n_obs} cells")
# print(f"  RNA: {mdata.mod['rna'].n_vars} genes")
# print(f"  ATAC: {mdata.mod['atac'].n_vars} regions")
# print(f"  TFs: {mdata.mod['rna'].var['is_tf'].sum()}")
# mdata

```{note}
The `read_legacy()` function loads data but does NOT apply the full schema validation.
For production use, follow up with `ds.pp.prepare_for_training()` to complete preprocessing.
```

## Step 2: Loading Legacy Models

Use `ds.tl.load_legacy_model()` to load a trained model from the legacy 3-file format.

**Requirements:**
- The 3 model checkpoint files (.pth)
- The data files (to reconstruct model architecture)
- The pre-computed E1 matrix (E1.pkl)

In [ ]:
# Set your paths
data_dir = "path/to/legacy/data"      # e.g., "/data/MM_lines"
results_dir = "path/to/legacy/results"  # e.g., "/results/MM_lines"

# Load legacy model
# model = ds.tl.load_legacy_model(
#     # Model checkpoints
#     vae_path=f"{results_dir}/best_model.pth",
#     tf2rnet_path=f"{results_dir}/best_model_tf2r.pth",
#     enformer_path=f"{results_dir}/best_model_tf2r_encoder.pth",
#     # Data files (needed to reconstruct architecture)
#     rna_path=f"{data_dir}/raw_exprMat_train.h5ad",
#     atac_path=f"{data_dir}/fragment_matrix_train.h5ad",
#     r2g_path=f"{data_dir}/r2gpenalty_train.npz",
#     tf_list_path=f"{data_dir}/TFs.txt",
#     # Pre-computed E1 matrix (required)
#     e1_path=f"{results_dir}/E1.pkl",
#     # Device
#     device="cpu",  # or "cuda"
# )
# 
# print(f"Model loaded!")
# print(f"  TFs: {len(model.tf_names)}")
# print(f"  Genes: {len(model.gene_names)}")
# print(f"  Regions: {len(model.region_names)}")
# print(f"  E1 shape: {model.adj_E1.shape}")

### Loading with PPI Network

If the legacy model was trained with PPI (protein-protein interaction) data, provide the PPI DataFrame:

In [ ]:
# Load PPI data (if used during training)
# ppi_data = pd.read_csv("path/to/ppi_interactions.csv")
# 
# model_with_ppi = ds.tl.load_legacy_model(
#     vae_path=f"{results_dir}/best_model.pth",
#     tf2rnet_path=f"{results_dir}/best_model_tf2r.pth",
#     enformer_path=f"{results_dir}/best_model_tf2r_encoder.pth",
#     rna_path=f"{data_dir}/raw_exprMat_train.h5ad",
#     atac_path=f"{data_dir}/fragment_matrix_train.h5ad",
#     r2g_path=f"{data_dir}/r2gpenalty_train.npz",
#     tf_list_path=f"{data_dir}/TFs.txt",
#     e1_path=f"{results_dir}/E1.pkl",
#     # PPI options
#     ppi_data=ppi_data,
#     ppi_confidence=0.5,
#     species="human",  # or "mouse"
# )

## Step 3: Loading GRN Matrices Only

If you only need the GRN matrices for analysis (without loading the full model), use `ds.tl.load_legacy_grn()`:

In [ ]:
# Load just the GRN matrices
# grn = ds.tl.load_legacy_grn(
#     e1_path=f"{results_dir}/E1.pkl",
#     e2_path=f"{results_dir}/E2.pkl",
# )
# 
# # E1: TF -> region binding matrix
# print(f"E1 shape: {grn['E1'].shape}")  # (n_tfs, n_regions)
# print(f"E1 columns (regions): {grn['E1'].columns[:5].tolist()}")
# print(f"E1 index (TFs): {grn['E1'].index[:5].tolist()}")
# 
# # E2: region -> gene regulatory matrix
# print(f"E2 shape: {grn['E2'].shape}")  # (n_regions, n_genes)
# print(f"E2 columns (genes): {grn['E2'].columns[:5].tolist()}")

This is useful for:
- Analyzing TF-target relationships without GPU
- Visualizing GRN networks
- Computing TF importance scores
- Comparing GRNs between conditions

## Step 4: Saving in New Format

Once loaded, save your data and model in the new unified format for easier future use.

### Saving Data

In [ ]:
# Save MuData to new format
# mdata.write("converted_data.h5mu")
# 
# # Verify by reloading
# mdata_reloaded = ds.read("converted_data.h5mu")
# print(f"Reloaded: {mdata_reloaded.n_obs} cells")

### Saving Model

In [ ]:
# Save model to new unified format
# model.save("converted_model.pt")
# 
# # Verify by reloading
# model_reloaded = ds.tl.load_model("converted_model.pt")
# print(f"Reloaded model with {len(model_reloaded.tf_names)} TFs")

```{note}
The new model format (~1GB) includes all components in a single file:
- VAE, MotifNet, and Enformer weights
- Pre-computed E1 matrix
- Feature names and training config

Future loading only requires: `model = ds.tl.load_model("model.pt")`
```

## Summary

| Task | Legacy | New |
|------|--------|-----|
| Load data | Multiple files | `ds.read("data.h5mu")` |
| Load model | 3 checkpoints + data files | `ds.tl.load_model("model.pt")` |
| Convert data | `ds.io.read_legacy(...)` | N/A |
| Convert model | `ds.tl.load_legacy_model(...)` | N/A |

## Next Steps

After converting your data and model:

- {doc}`03_grn_analysis` - Analyze the gene regulatory network
- {doc}`04_perturbation_simulation` - Simulate TF perturbations